<a href="https://colab.research.google.com/github/saran7222/My-Portfolio/blob/main/Research_Paper_Answer_Bot_MAIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Research Paper Answer Bot

**Before running (Runtime → Restart session, then Run all):**
1. Add a Colab secret named `groq_api_key` (key icon in the left sidebar) with your Groq API key, and enable notebook access for it.
2. If you plan to deploy the Streamlit app via ngrok (last section), also add a secret named `NGROK_TOKEN`.
3. In your Google Drive, make sure this folder exists and contains your PDFs: `MyDrive/Research_Paper_Answer_Bot/Research_Papers`. You'll be prompted to authorize Drive access when the imports cell runs.


## 1. Setup — Install & Import

In [1]:
# One consolidated install — pinned versions that are known to work together.
# NOTE: everything is installed in a SINGLE pip call so pip's dependency resolver
# sees all constraints at once (numpy<2.0 included) instead of pinning numpy in
# one pass and letting a later, separate pip call silently upgrade it again.
!pip uninstall -y numpy sentence-transformers -q
!pip install -q "numpy<2.0" sentence-transformers \
    langchain==0.2.16 langchain-community==0.2.16 langchain-huggingface==0.0.3 \
    langchain-groq==0.1.9 chromadb==0.5.5 pypdf groq streamlit pyngrok
print("Installed. If this is the FIRST time running this cell, go to Runtime -> Restart session, then Run all again.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.11.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.8.23 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.1 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which i

In [2]:
import os
from google.colab import drive
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata
from groq import Groq

# This was missing before: pdf_folder (below) lives under /content/drive/...
# but the Drive was never actually mounted, so a fresh "Run all" would fail
# with FileNotFoundError as soon as it tried to list pdf_folder.
drive.mount('/content/drive')

print("Imports OK")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports OK


## 2. Load Research Papers

Update `pdf_folder` to match where your PDFs live in Google Drive.

In [3]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# OPTIONAL: only run this manually if you want to upload extra PDFs directly
# into this Colab session instead of using Google Drive. It was previously
# left in the pipeline unused (pdf_folder below reads from Drive, not from
# this upload), and calling files.upload() during an unattended "Run all"
# would block forever waiting for a file picker. Set RUN_MANUAL_UPLOAD=True
# to use it.
RUN_MANUAL_UPLOAD = False

if RUN_MANUAL_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Skipped manual upload — using Google Drive folder instead.")


Skipped manual upload — using Google Drive folder instead.


In [5]:
pdf_folder = "/content/drive/MyDrive/Research_Paper_Answer_Bot/Research_Paper."

if not os.path.isdir(pdf_folder):
    raise FileNotFoundError(
        f"Could not find {pdf_folder}. Make sure Google Drive is mounted "
        "(see the cell above) and that this exact folder exists in your Drive, "
        "or update pdf_folder to point at your own PDFs."
    )

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
print("Total PDF Files:", len(pdf_files))
print(pdf_files)

if not pdf_files:
    raise ValueError(f"No PDF files found in {pdf_folder}.")

Total PDF Files: 7
['2005.11401v4.pdf', '2407.21059v1.pdf', '2401.15884v3.pdf', '2307.09288v2.pdf', '1908.10084v1.pdf', '2407.12994v2.pdf', '1706.03762v7.pdf']


In [6]:
all_documents = []
for file in pdf_files:
    path = os.path.join(pdf_folder, file)
    loader = PyPDFLoader(path)
    docs = loader.load()
    all_documents.extend(docs)

print("Total Pages Loaded:", len(all_documents))

Total Pages Loaded: 194


## 3. Chunking Strategy

Using `RecursiveCharacterTextSplitter` with `chunk_size=1000`, `chunk_overlap=200` — large enough to keep a full
idea together, with 20% overlap so context isn't lost right at a chunk boundary. Metadata (`source`, `page`) is
preserved automatically by `PyPDFLoader`, which we use later for citing sources.

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_chunks = splitter.split_documents(all_documents)
print("Total Chunks:", len(all_chunks))
print("\nSample chunk metadata:", all_chunks[0].metadata)

Total Chunks: 889

Sample chunk metadata: {'source': '/content/drive/MyDrive/Research_Paper_Answer_Bot/Research_Paper./2005.11401v4.pdf', 'page': 0}


## 4. Embedding Models — Compare At Least Two

Comparing an open-source **all-MiniLM-L6-v2** (fast, 384-dim) against **BAAI/bge-small-en-v1.5**
(stronger semantic matching, also 384-dim) on the same chunk set.

In [8]:
# Model 1: all-MiniLM-L6-v2
# collection_name is set explicitly (and different from Model 2's) so the two
# comparisons never end up sharing one Chroma collection.
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_database = Chroma.from_documents(
    all_chunks, embedding_model, collection_name="minilm_l6_v2"
)
print("Model 1 (all-MiniLM-L6-v2) indexed.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Model 1 (all-MiniLM-L6-v2) indexed.


In [9]:
# Model 2: bge-small-en-v1.5
# Separate collection_name from Model 1 above — see note in the previous cell.
embedding_model_2 = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vector_database_2 = Chroma.from_documents(
    all_chunks, embedding_model_2, collection_name="bge_small_en_v1_5"
)
print("Model 2 (bge-small-en-v1.5) indexed.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Model 2 (bge-small-en-v1.5) indexed.


In [10]:
test_question = "What is Modular RAG?"

def _page_display(meta):
    page = meta.get("page", 0)
    return (page + 1) if isinstance(page, int) else "?"

print("=== MiniLM (Model 1) Results ===")
docs_1 = vector_database.similarity_search(test_question, k=3)
for i, d in enumerate(docs_1, start=1):
    print(f"{i}. {os.path.basename(d.metadata.get('source'))} (Page {_page_display(d.metadata)})")
    print(d.page_content[:200], "\n")

print("\n=== bge-small (Model 2) Results ===")
docs_2 = vector_database_2.similarity_search(test_question, k=3)
for i, d in enumerate(docs_2, start=1):
    print(f"{i}. {os.path.basename(d.metadata.get('source'))} (Page {_page_display(d.metadata)})")
    print(d.page_content[:200], "\n")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


=== MiniLM (Model 1) Results ===
1. 2407.21059v1.pdf (Page 3)
specific functions or tasks. This architecture is divided into
three levels: the top level focuses on the critical stages of
RAG, where each stage is treated as an independent module.
This level not o 

2. 2407.21059v1.pdf (Page 9)
process design. A RAG flow pattern can be defined as P =
{Mϕ1 :{Op1}→ Mϕ2 :{Op2}→ ... →Mϕn :{Opn}}
A. Linear Pattern
The modules in the modular RAG system are organized in
a linear way, and can be des 

3. 2407.21059v1.pdf (Page 15)
seamless adaptation to the dynamic requirements of various
applications.
A. Opportunities in Modular RAG
The benefits of Modular RAG are evident, providing a
fresh and comprehensive perspective on exi 


=== bge-small (Model 2) Results ===
1. 2407.21059v1.pdf (Page 16)
16
VII. C ONCLUSION
RAG is emerging as a pivotal technology for LLM applica-
tions. As technological landscapes evolve and the intricacies of
application requirements escalate, RAG systems are being e 



## 5. Retrieval Strategy — Compare At Least Two

Comparing baseline **Similarity Search (cosine)** against **MMR (Max Marginal Relevance)**, which balances
relevance with diversity in the retrieved chunks.

In [11]:
print("=== Similarity Search (Cosine) ===")
docs_similarity = vector_database.similarity_search(test_question, k=3)
for i, d in enumerate(docs_similarity, start=1):
    print(f"{i}. {os.path.basename(d.metadata.get('source'))} (Page {_page_display(d.metadata)})")

print("\n=== MMR (Max Marginal Relevance) ===")
docs_mmr = vector_database.max_marginal_relevance_search(test_question, k=3, fetch_k=10)
for i, d in enumerate(docs_mmr, start=1):
    print(f"{i}. {os.path.basename(d.metadata.get('source'))} (Page {_page_display(d.metadata)})")


=== Similarity Search (Cosine) ===
1. 2407.21059v1.pdf (Page 3)
2. 2407.21059v1.pdf (Page 9)
3. 2407.21059v1.pdf (Page 15)

=== MMR (Max Marginal Relevance) ===
1. 2407.21059v1.pdf (Page 3)
2. 2407.21059v1.pdf (Page 9)
3. 2407.21059v1.pdf (Page 15)


In [12]:
api_key = userdata.get("groq_api_key")
if not api_key:
    raise ValueError(
        'Colab secret "groq_api_key" is not set. Click the key icon in the left '
        'sidebar, add a secret named groq_api_key with your Groq API key, and '
        'make sure notebook access is enabled for it.'
    )
client = Groq(api_key=api_key)
print("Connected to Groq")


Connected to Groq


In [13]:
def create_prompt(q, docs):
    text = ""
    for d in docs:
        text += d.page_content + "\n"

    return f"""
You are a Research Paper Assistant.

Answer only from the given context.

If the answer is not available, reply with EXACTLY this sentence and nothing else:
"I don't know based on the provided documents."

Context:
{text}

Question:
{q}
"""

 6. RAG Pipeline Connect retriever to **LLM**

In [14]:
def get_answer_with_sources(q, vector_db=None, k=3):
    vector_db = vector_db or vector_database
    docs = vector_db.similarity_search(q, k=k)
    prompt = create_prompt(q, docs)
    reply = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
    )
    answer = reply.choices[0].message.content

    sources = []
    for d in docs:
        source_file = d.metadata.get("source", "Unknown")
        page = d.metadata.get("page", 0)
        page_display = (page + 1) if isinstance(page, int) else "?"
        sources.append(f"{os.path.basename(source_file)} (Page {page_display})")

    return answer, sources

In [15]:
api_key = userdata.get("groq_api_key")
if not api_key:
    raise ValueError(
        'Colab secret "groq_api_key" is not set. Click the key icon in the left '
        'sidebar, add a secret named groq_api_key with your Groq API key, and '
        'make sure notebook access is enabled for it.'
    )
client = Groq(api_key=api_key)
print("Connected to Groq")

Connected to Groq


In [16]:
question = "What is Modular RAG?"
answer, sources = get_answer_with_sources(question)

print("ANSWER:")
print(answer)
print("\nTOP-3 SOURCES:")
for i, s in enumerate(sources, start=1):
    print(f"{i}. {s}")

ANSWER:
Modular RAG is an architectural framework that structures Retrieval‑Augmented Generation (RAG) systems into three hierarchical levels: a top level with independent modules and an orchestration module that coordinates RAG processes; a middle level composed of sub‑modules that refine and optimize functions; and a bottom level of basic operators that perform the elementary operations. It represents RAG as a computational graph, where nodes are specific operators, and it builds on earlier RAG paradigms (Advanced RAG and Naive RAG) by adding modularity, scalability, and flexibility for designing, implementing, and extending RAG flows.

TOP-3 SOURCES:
1. 2407.21059v1.pdf (Page 3)
2. 2407.21059v1.pdf (Page 9)
3. 2407.21059v1.pdf (Page 15)


## 7. Testing & Evaluation

Running 10 test questions spanning different papers, and checking each answer for relevance, accuracy,
and groundedness.

In [17]:
test_questions = [
    "What is Modular RAG?",
    "What is the difference between Naive RAG and Advanced RAG?",
    "What are the main modules in Modular RAG architecture?",
    "What is a RAG flow pattern?",
    "What is the Linear Pattern in Modular RAG?",
    "What is the Transformer architecture?",
    "What is self-attention in the Transformer model?",
    "What is multi-head attention?",
    "How does positional encoding work in Transformers?",
    "What are the advantages of using attention mechanisms over recurrence?",
]

for i, q in enumerate(test_questions, start=1):
    answer, sources = get_answer_with_sources(q)
    print(f"Q{i}: {q}")
    print(f"Answer: {answer}")
    print(f"Sources: {sources}")
    print("-" * 80)

Q1: What is Modular RAG?
Answer: Modular RAG is an architecture that decomposes Retrieval‑Augmented Generation (RAG) systems into three hierarchical levels—top, middle, and bottom—each handling different granularities of the RAG process. The top level treats each critical stage of RAG as an independent module and adds an orchestration module to coordinate these stages. The middle level consists of sub‑modules that refine and optimize the functions of each stage, while the bottom level contains the basic operational units called operators. In this framework, RAG systems can be represented as computational graphs where nodes are operators. Modular RAG builds on earlier paradigms, with Advanced RAG being a special case and Naive RAG a further special case, and it emphasizes modularity, scalability, and adaptability for research and application development.
Sources: ['2407.21059v1.pdf (Page 3)', '2407.21059v1.pdf (Page 9)', '2407.21059v1.pdf (Page 15)']
------------------------------------

8.Streamlit app - Strech Goals

In [18]:
%%writefile app.py

import streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import tempfile
import os
import torch

st.set_page_config(page_title="Research Paper Answer Bot", page_icon="📄", initial_sidebar_state="collapsed")

st.title("Research Paper Question Answer Bot")
st.caption("Ask questions from your research papers")

# hide the user avatar bubble + make chat input full width
st.markdown("""
<style>
.user-msg {
    text-align: right;
    background-color: #2b313e;
    padding: 8px 14px;
    border-radius: 10px;
    margin-bottom: 8px;
}
div[data-testid="stChatInput"] {
    width: 100%;
}
</style>
""", unsafe_allow_html=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@st.cache_resource
def load_embedding_model(model_name):
    return HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": DEVICE})

groq_api_key = st.sidebar.text_input("Enter Groq API Key", type="password")

embedding_choice = st.sidebar.selectbox(
    "Embedding model",
    ["all-MiniLM-L6-v2 (fast)", "bge-small-en-v1.5 (better quality)"]
)
EMBED_MAP = {
    "all-MiniLM-L6-v2 (fast)": "sentence-transformers/all-MiniLM-L6-v2",
    "bge-small-en-v1.5 (better quality)": "BAAI/bge-small-en-v1.5",
}

retrieval_choice = st.sidebar.selectbox(
    "Retrieval strategy",
    ["Similarity (cosine)", "MMR (diversity)"]
)

if "uploader_key" not in st.session_state:
    st.session_state["uploader_key"] = 0

if st.sidebar.button("🗑️ Clear chat history"):
    st.session_state["chat_history"] = []
    st.session_state.pop("vector_db", None)
    st.session_state.pop("uploaded_names", None)
    st.session_state.pop("embedding_used", None)
    st.session_state["uploader_key"] += 1   # changes the uploader's key so it resets
    st.rerun()

uploaded_files = st.file_uploader(
    "Upload Research Papers",
    type="pdf",
    accept_multiple_files=True,
    key=f"pdf_uploader_{st.session_state['uploader_key']}"
)

if "chat_history" not in st.session_state:
    st.session_state["chat_history"] = []

if uploaded_files and groq_api_key:

    names_changed = st.session_state.get("uploaded_names") != [f.name for f in uploaded_files]
    embed_changed = st.session_state.get("embedding_used") != embedding_choice

    if "vector_db" not in st.session_state or names_changed or embed_changed:
        with st.spinner("Indexing research papers..."):
            all_documents = []
            for uploaded_file in uploaded_files:
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_file:
                    temp_file.write(uploaded_file.read())
                    pdf_path = temp_file.name

                try:
                    loader = PyPDFLoader(pdf_path)
                    documents = loader.load()
                    for doc in documents:
                        doc.metadata["source"] = uploaded_file.name
                    all_documents.extend(documents)
                finally:
                    os.remove(pdf_path)

            if not all_documents:
                st.error("Could not extract any text from the uploaded PDF(s).")
                st.stop()

            splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            chunks = splitter.split_documents(all_documents)

            embedding = load_embedding_model(EMBED_MAP[embedding_choice])

            vector_db = Chroma.from_documents(
                documents=chunks,
                embedding=embedding,
                collection_name=f"session_{abs(hash(embedding_choice))}",
            )

            st.session_state["vector_db"] = vector_db
            st.session_state["uploaded_names"] = [f.name for f in uploaded_files]
            st.session_state["embedding_used"] = embedding_choice
            st.session_state["chat_history"] = []

        st.success(f"Indexed {len(chunks)} chunks from {len(uploaded_files)} paper(s) using **{embedding_choice}**.")

    vector_db = st.session_state["vector_db"]

    search_type = "mmr" if retrieval_choice.startswith("MMR") else "similarity"
    search_kwargs = {"k": 6, "fetch_k": 12} if search_type == "mmr" else {"k": 6}
    retriever = vector_db.as_retriever(search_type=search_type, search_kwargs=search_kwargs)

    llm = ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-20b", temperature=0)

    prompt = ChatPromptTemplate.from_template("""
You are a helpful research paper assistant having a conversation with a user.
Use only the given context to answer the question.
If needed, use the chat history to understand follow-up questions (e.g. "it", "that paper", "the same one").

If the answer is not found in the context, reply with EXACTLY this sentence and nothing else:
"I don't know based on the provided documents."

Chat History:
{chat_history}

Context:
{context}

Question:
{question}

Answer:
""")

    for turn in st.session_state["chat_history"]:
        st.markdown(f'<div class="user-msg">{turn["question"]}</div>', unsafe_allow_html=True)
        with st.chat_message("assistant"):
            st.write(turn["answer"])

    question = st.chat_input("Ask a question about your papers")

    if question:
        st.markdown(f'<div class="user-msg">{question}</div>', unsafe_allow_html=True)

        history_text = ""
        for turn in st.session_state["chat_history"][-3:]:
            history_text += f"User: {turn['question']}\nAssistant: {turn['answer']}\n\n"

        docs = retriever.invoke(question)
        context = "\n\n".join([doc.page_content for doc in docs])
        messages = prompt.invoke({"chat_history": history_text, "context": context, "question": question})

        try:
            response = llm.invoke(messages)
            answer = response.content
        except Exception as e:
            answer = f"Error calling Groq API: {e}"

        with st.chat_message("assistant"):
            st.write(answer)

            answer_lower = answer.lower()
            not_found_phrases = [
                "i don't know",
                "i don't see",
                "i cannot find",
                "i couldn't find",
                "not mentioned",
                "no mention of",
                "not found in the context",
                "not found in the provided",
                "error calling groq api",
            ]
            found_answer = not any(phrase in answer_lower for phrase in not_found_phrases)

            if found_answer:
                st.markdown("**Top-3 Supporting Sources**")
                seen = set()
                shown = 0
                for doc in docs:
                    page = doc.metadata.get("page", 0)
                    page_display = page + 1 if isinstance(page, int) else page
                    source = doc.metadata.get("source")
                    key = (source, page_display)
                    if key in seen:
                        continue
                    seen.add(key)
                    shown += 1
                    st.write(f"{shown}. **{source}** — Page {page_display}")
                    if shown == 3:
                        break

        st.session_state["chat_history"].append({"question": question, "answer": answer})

else:
    st.info("Upload at least one PDF and enter your Groq API key to begin.")

Overwriting app.py


Deploy the App


In [19]:
from pyngrok import ngrok
import subprocess, time

ngrok.kill()
!pkill -9 -f streamlit 2>/dev/null
time.sleep(2)

ngrok_token = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(ngrok_token)

streamlit_process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(8)

public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)

^C
Your app is live at: NgrokTunnel: "https://mousiness-engross-popsicle.ngrok-free.dev" -> "http://localhost:8501"
